# N047 · 快慢指针、环与中点

**目标：** 解释指针速度关系并处理不同中点约定。

**先修：** N045, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Floyd相遇；环入口推导；奇偶长度；倒数第K个。

## 从问题到算法

快慢指针速度差为1。进入环后，相对位移每轮增加1，因此有限轮内相遇；无环则快指针先到None。设头到入口长度为μ、环长为λ，相遇时慢指针走过的步数是λ的倍数；一指针回到头，两者同速前进会在入口相遇。偶数长度的中点返回第二个中点。

## 最小实现

**本章接口：** `has_cycle(head)`、`detect_cycle(head)`、`middle_node(head)`、`remove_nth_from_end(head, n)`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

def has_cycle(head):
    slow=fast=head
    while fast and fast.next:
        slow=slow.next; fast=fast.next.next
        if slow is fast: return True
    return False

def detect_cycle(head):
    slow=fast=head
    while fast and fast.next:
        slow=slow.next; fast=fast.next.next
        if slow is fast:
            slow=head
            while slow is not fast: slow=slow.next; fast=fast.next
            return slow
    return None

def middle_node(head):
    slow=fast=head
    while fast and fast.next: slow=slow.next; fast=fast.next.next
    return slow

def remove_nth_from_end(head,n):
    if n<1: raise ValueError('n must be positive')
    dummy=ListNode(0,head); fast=slow=dummy
    for _ in range(n):
        fast=fast.next
        if fast is None: raise ValueError('n exceeds length')
    while fast.next: fast=fast.next; slow=slow.next
    slow.next=slow.next.next
    return dummy.next

## 正确性、前提与复杂度

入口定位使用模λ的距离关系，不依赖节点值。倒数删除保持fast比slow领先n个next边；fast到末节点时slow恰是删除目标前驱。

**代价：** 均 O(n)时间、O(1)辅助空间；有环时n表示可达不同节点数。中点与删除要求无环输入。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['长度','第二中点值'],[(n,middle_node(list_to_nodes(list(range(n)))).val) for n in range(1,9)])

长度,第二中点值
1,0
2,1
3,1
4,2
5,2
6,3
7,3
8,4


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
h=list_to_nodes([1,2,3,4]); entry=h.next; h.next.next.next.next=entry
assert has_cycle(h) and detect_cycle(h) is entry
h=ListNode(9); h.next=h; assert detect_cycle(h) is h
assert not has_cycle(list_to_nodes([1,2])) and detect_cycle(None) is None
assert middle_node(list_to_nodes([1,2,3,4])).val==3
assert nodes_to_list(remove_nth_from_end(list_to_nodes([1,2,3]),3))==[2,3]
assert nodes_to_list(remove_nth_from_end(list_to_nodes([1,2,3]),1))==[1,2]
assert remove_nth_from_end(ListNode(1),1) is None
print('N047: 所有本章断言通过')

N047: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 推导相遇后回到头部的理由。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分偶数长度前中点和后中点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 141. Linked List Cycle（canonical）
- 142. Linked List Cycle II（canonical）
- 876. Middle of the Linked List（canonical）
- 19. Remove Nth Node From End of List（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。